In [1]:
# =====================================================================
# STEP 1: DOWNLOAD AND DATA PREPARATION
# =====================================================================
import os
import urllib.request
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
filename = "winequality-red.csv"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

df = pd.read_csv(filename, delimiter=';')
df.drop_duplicates(inplace=True)
df.columns = df.columns.str.replace(' ', '_')

# Binarizing the target variable:
# Quality scores 3-5 are labeled '0' (Low Quality), scores 6-8 labeled '1' (High Quality)
df['quality_label'] = df['quality'].apply(lambda x: 1 if x >= 6 else 0)

# Split features and label
X = df.drop(['quality', 'quality_label'], axis=1)
y = df['quality_label']

# =====================================================================
# STEP 2: MODEL TRAINING AND EVALUATION
# =====================================================================

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Normalize the features using standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train a Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Run Predictions
y_pred = model.predict(X_test_scaled)

# Output evaluation metrics
print(f"Model Global Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("--- Classification Metric Report ---")
print(classification_report(y_test, y_pred, target_names=['Low Quality (0-5)', 'High Quality (6-8)']))

# Feature Importance Tracking
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("--- Top 3 Most Influential Features for Classification ---")
print(importance.head(3))

Model Global Accuracy: 76.84%

--- Classification Metric Report ---
                    precision    recall  f1-score   support

 Low Quality (0-5)       0.74      0.79      0.76       128
High Quality (6-8)       0.80      0.75      0.77       144

          accuracy                           0.77       272
         macro avg       0.77      0.77      0.77       272
      weighted avg       0.77      0.77      0.77       272

--- Top 3 Most Influential Features for Classification ---
alcohol             0.213198
sulphates           0.121831
volatile_acidity    0.108305
dtype: float64
